# Tutoriel 7

[Télécharger le tutoriel](../07_tutoriel.zip)

# Affichage interactif de résultats 2D en temps réel

Ce tutoriel explique comment produire des **figures 2D de qualité** pour afficher les résultats d'un modèle en temps réel : échelle de couleur fixe, coordonnées physiques, proportions correctes, labels avec unités et temps arrondi dans le titre.

## 1. Les éléments essentiels

La figure doit être créée **une seule fois, avant la boucle**, avec sa colorbar. Ensuite, cinq réglages font toute la différence :

### ✓ 1. Échelle de couleur fixe
Sans bornes fixes, l'échelle se réajuste à chaque itération : une même couleur ne représente plus la même valeur d'une image à l'autre, et l'animation devient impossible à interpréter. On fixe les bornes avec `ax.imshow(..., vmin=5, vmax=15)`, ou après coup avec `s.set_clim(5, 15)`.

> ⚠️ **Piège classique** : si vous appelez `ax.cla()` dans la boucle, l'image est détruite et recréée à chaque affichage. Il faut donc **repasser `vmin` et `vmax` à chaque `imshow`**, faute de quoi la colorbar — elle, créée une seule fois avant la boucle — ne correspondra plus à ce qui est affiché.

### ✓ 2. `extent` pour les coordonnées physiques
`extent=[xmin, xmax, ymin, ymax]` affiche les vraies coordonnées (m, km) au lieu des indices de la grille.

### ✓ 3. `origin='lower'`
Place l'origine (0,0) en bas à gauche, comme dans un repère cartésien. Sans lui, le champ est affiché à l'envers.

### ✓ 4. `ax.set_aspect('equal')`
Conserve les proportions du domaine, sans quoi un domaine de 100 × 80 m apparaît déformé.

### ✓ 5. Labels avec unités et temps arrondi
Toujours indiquer les unités. Et sans arrondi, le titre affiche quinze décimales : écrivez `'t = ' + str(round(temps, 1)) + ' ans'`.

## 2. Le code complet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

Lx = 100.0      # longueur du domaine, m
Ly = 80.0       # largeur du domaine, m
nx = 50         # nombre de points de grille en x
ny = 40         # nombre de points de grille en y
X, Y   = np.meshgrid(np.linspace(0,Lx,nx), np.linspace(0,Ly,ny))

nt          = 100     # nombre d'iterations
freq_affich = 10      # frequence d'affichage
dt          = 0.5     # pas de temps, ans

# figure ET colorbar creees AVANT la boucle
fig, ax = plt.subplots(figsize=(6.5,4.5))
s    = ax.imshow(0*X, extent=[0,Lx,0,Ly], origin='lower', cmap='jet', vmin=5, vmax=15)
cbar = plt.colorbar(s, ax=ax)
cbar.set_label("Température (°C)", rotation=270, labelpad=20)

temps = 0.0
for it in range(nt):

    T = 10 + 5*np.sin(2*np.pi*X/Lx + 0.1*it)*np.cos(2*np.pi*Y/Ly)   # le "modele"
    temps += dt

    if it % freq_affich == 0:
        clear_output(wait=True)
        ax.cla()                       # efface les axes, PAS la colorbar
        ax.imshow(T, extent=[0,Lx,0,Ly], origin='lower', cmap='jet',
                  vmin=5, vmax=15)     # vmin/vmax a repasser, voir le piege ci-dessus
        ax.set_aspect('equal')
        ax.set_xlabel('Distance horizontale x (m)', fontsize=12)
        ax.set_ylabel('Distance verticale y (m)', fontsize=12)
        ax.set_title('Température au temps t = ' + str(round(temps, 1)) + ' années', fontsize=14)
        display(fig)
        plt.pause(0.02)

plt.close()

## 3. Checklist d'une figure de qualité

| Élément | Commande | ✓ |
|---|---|---|
| **Échelle de couleur fixe** | `imshow(..., vmin=..., vmax=...)` | □ |
| **Coordonnées physiques** | `extent=[xmin, xmax, ymin, ymax]` | □ |
| **Origine en bas** | `origin='lower'` | □ |
| **Proportions** | `ax.set_aspect('equal')` | □ |
| **Titre avec temps arrondi** | `str(round(temps, 1))` | □ |
| **Labels des axes avec unité** | `ax.set_xlabel('Distance (m)')` | □ |
| **Label de colorbar avec unité** | `cbar.set_label('Température (°C)')` | □ |
| **Taille de police lisible** | `fontsize=12` ou `14` | □ |

## À expérimenter

Dégradez le code ci-dessus, **une modification à la fois**, et regardez ce que chacune coûte :

1. enlevez `vmin=5, vmax=15` de l'`imshow` **de la boucle** : la colorbar correspond-elle encore aux couleurs affichées ?
2. enlevez `origin='lower'`, puis `extent`, puis `ax.set_aspect('equal')`.